In [11]:
import pandas as pd
import numpy as np
from mlforecast import MLForecast
import sys
import os
sys.path.append(os.path.abspath("../.."))
from mlforecast.lag_transforms import RollingMean, RollingStd
import lightgbm as lgb
import holidays
from tinyshift.modelling import ISSMForecastWrapper, ISSMForecastEvaluator

In [12]:
def generate_m5_simulated_data(n_stores=3, n_skus=5, start_date="2023-01-01", days=365, seed=42):

    np.random.seed(seed)
    dates = pd.date_range(start=start_date, periods=days, freq="D")
    n_days = len(dates)
    
    data_list = []

    for store_id in range(1, n_stores + 1):
        store_name = f"STORE_{store_id:02d}"
        for sku_id in range(1, n_skus + 1):
            item_id = f"FOODS_1_{sku_id:03d}"
            
            base_demand = np.random.uniform(0.5, 5.0)
            
            dow_factor = np.tile([0.8, 0.85, 0.9, 0.95, 1.1, 1.4, 1.3], int(np.ceil(n_days/7)))[:n_days]
            
            base_price = np.random.uniform(2.0, 15.0)
            prices = base_price * np.random.choice([1.0, 0.85, 0.70], size=n_days, p=[0.8, 0.15, 0.05])
            price_elasticity = np.exp(-0.15 * (prices - base_price))
            
            event_indices = np.random.choice(n_days, size=12, replace=False)
            event_impact = np.ones(n_days)
            event_impact[event_indices] = np.random.uniform(1.3, 2.2, size=12)
            
            lambda_t = base_demand * dow_factor * price_elasticity * event_impact
            
            sales = np.random.poisson(lambda_t)
            
            for d_idx, d_date in enumerate(dates):
                data_list.append({
                    'date': d_date,
                    'store_id': store_name,
                    'item_id': item_id,
                    'sales': sales[d_idx],
                    'sell_price': round(prices[d_idx], 2),
                    'is_event': 1 if d_idx in event_indices else 0,
                })

    return pd.DataFrame(data_list)

df_raw = generate_m5_simulated_data(n_stores=3, n_skus=5, days=365)

df_raw['unique_id'] = df_raw['store_id'] + '_' + df_raw['item_id']

df_nixtla = df_raw.rename(columns={
    'date': 'ds',
    'sales': 'y'
})

In [13]:
def temporal_split_by_horizon(df, time_col='ds', horizon_days=28):
    df = df.sort_values(time_col)
    max_date = df[time_col].max()
    cutoff_date = max_date - pd.Timedelta(days=horizon_days)
    
    df_train = df[df[time_col] <= cutoff_date].copy()
    df_test = df[df[time_col] > cutoff_date].copy()
    
    return df_train, df_test, cutoff_date

In [14]:
us_holidays = holidays.US(years=range(2022, 2027))

_extended_holiday_dates = set()
for h_date in us_holidays.keys():
    h_timestamp = pd.Timestamp(h_date)
    for offset in range(-2, 1):  # Véspera (-2, -1) e o próprio dia (0)
        _extended_holiday_dates.add((h_timestamp + pd.Timedelta(days=offset)).date())


def is_holiday_window(dates) -> pd.Series:
    dates_series = pd.Series(dates)
    return dates_series.dt.date.isin(_extended_holiday_dates).astype(int)

def add_relative_price(df: pd.DataFrame, id_col: str = 'unique_id', price_col: str = 'sell_price') -> pd.DataFrame:
    df = df.copy()
    
    mean_price_per_sku = df.groupby(id_col)[price_col].transform('mean')
    
    df['relative_price'] = df[price_col] / (mean_price_per_sku + 1e-6)
    
    return df

df_nixtla['is_holiday_window'] = is_holiday_window(df_nixtla['ds'])
df_nixtla = add_relative_price(df_nixtla)

fcst = MLForecast(
    models={
        'lgb_issm': lgb.LGBMRegressor(
            objective='poisson',
            metric='rmse',
            n_estimators=100,
            learning_rate=0.05,
            random_state=42,
            verbosity=-1
        )
    },
    freq='D',
    lags=[7, 14, 28],
    lag_transforms={
        1: [RollingMean(window_size=7), RollingStd(window_size=7)],
    },
    date_features=['dayofweek', 'month', 'dayofyear']
)

In [15]:
df_train, df_test, cutoff = temporal_split_by_horizon(df_nixtla, horizon_days=28)

issm = ISSMForecastWrapper(fcst)
issm.fit(df_train[["ds", "y", "sell_price", 'relative_price', "is_event", "is_holiday_window", "unique_id"]], gamma=0.7)

In [16]:
df_test[["ds", "sell_price", "is_event", "is_holiday_window", "unique_id"]].groupby("unique_id")["ds"].nunique()

unique_id
STORE_01_FOODS_1_001    28
STORE_01_FOODS_1_002    28
STORE_01_FOODS_1_003    28
STORE_01_FOODS_1_004    28
STORE_01_FOODS_1_005    28
STORE_02_FOODS_1_001    28
STORE_02_FOODS_1_002    28
STORE_02_FOODS_1_003    28
STORE_02_FOODS_1_004    28
STORE_02_FOODS_1_005    28
STORE_03_FOODS_1_001    28
STORE_03_FOODS_1_002    28
STORE_03_FOODS_1_003    28
STORE_03_FOODS_1_004    28
STORE_03_FOODS_1_005    28
Name: ds, dtype: int64

In [17]:
df_res = issm.pmf(h=28, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]], max_k=4)
df_res.loc[:, "y"] = df_test["y"].values

In [18]:
df_res

,unique_id,ds,lambda_t,r_dispersion,P(Y=0),P(Y=1),P(Y=2),P(Y=3),P(Y=4),P(Y>4),y
0,STORE_01_FOODS_1_001,2023-12-04,1.629409,50.0,0.201207,0.317501,0.255516,0.139776,0.058450,0.027550,1
1,STORE_01_FOODS_1_001,2023-12-05,1.620119,50.0,0.203025,0.318602,0.254986,0.138716,0.057686,0.026985,5
2,STORE_01_FOODS_1_001,2023-12-06,2.222906,50.0,0.113618,0.241812,0.262469,0.193652,0.109219,0.079230,1
3,STORE_01_FOODS_1_001,2023-12-07,2.850100,50.0,0.062547,0.168653,0.231925,0.216793,0.154908,0.165173,3
4,STORE_01_FOODS_1_001,2023-12-08,3.506878,50.0,0.033730,0.110535,0.184736,0.209867,0.182252,0.278879,3
...,...,...,...,...,...,...,...,...,...,...,...
415,STORE_03_FOODS_1_005,2023-12-27,3.538167,50.0,0.032759,0.108246,0.182417,0.208959,0.182975,0.284644,4
416,STORE_03_FOODS_1_005,2023-12-28,4.126887,50.0,0.018960,0.072282,0.140533,0.185724,0.187626,0.394875,0
417,STORE_03_FOODS_1_005,2023-12-29,5.009677,50.0,0.008444,0.038449,0.089289,0.140945,0.170073,0.552800,2
418,STORE_03_FOODS_1_005,2023-12-30,4.801258,50.0,0.010209,0.044721,0.099911,0.151726,0.176133,0.517301,2


In [19]:
df_res = issm.predict(h=28, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]], quantiles=[0.05, 0.50, 0.95, 0.99])
df_res.loc[:, "y"] = df_test["y"].values

In [20]:
ISSMForecastEvaluator.evaluate(df_res, quantiles=[0.05, 0.50, 0.95, 0.99])

,Pinball Loss,Target Coverage,Empirical Coverage,Coverage Gap
q_5,0.2008,0.05,0.3000,0.2500
q_50,1.0810,0.50,0.6095,0.1095
q_95,0.5005,0.95,0.8548,-0.0952
q_99,0.2247,0.99,0.9262,-0.0638
